# Lab4-Assignment about Named Entity Recognition and Classification

This notebook describes the assignment of Lab 4 of the text mining course. We assume you have succesfully completed Lab1, Lab2 and Lab3 as welll. Especially Lab2 is important for completing this assignment.

**Learning goals**
* going from linguistic input format to representing it in a feature space
* working with pretrained word embeddings
* train a supervised classifier (SVM)
* evaluate a supervised classifier (SVM)
* learn how to interpret the system output and the evaluation results
* be able to propose future improvements based on the observed results


## Credits
This notebook was originally created by [Marten Postma](https://martenpostma.github.io) and [Filip Ilievski](http://ilievski.nl) and adapted by Piek vossen

## [Points: 18] Exercise 1 (NERC): Training and evaluating an SVM using CoNLL-2003

**[4 point] a) Load the CoNLL-2003 training data using the *ConllCorpusReader* and create for both *train.txt* and *test.txt*:**

    [2 points]  -a list of dictionaries representing the features for each training instances, e..g,
    ```
    [
    {'words': 'EU', 'pos': 'NNP'}, 
    {'words': 'rejects', 'pos': 'VBZ'},
    ...
    ]
    ```

    [2 points] -the NERC labels associated with each training instance, e.g.,
    dictionaries, e.g.,
    ```
    [
    'B-ORG', 
    'O',
    ....
    ]
    ```

In [12]:
from nltk.corpus.reader import ConllCorpusReader                                                                                                                
### Adapt the path to point to the CONLL2003 folder on your local machine
train = ConllCorpusReader('CONLL2003/CONLL2003', 'train.txt', ['words', 'pos', 'ignore', 'chunk'])                                                              
                                                                                                                                                                
training_features = []                                                                                                                                          
training_gold_labels = []                                                                                                                                       
                
for token, pos, ne_label in train.iob_words():
    a_dict = {
        'words': token,
        'pos': pos,
    }                                                                                                                                                           
    training_features.append(a_dict)
    training_gold_labels.append(ne_label)                                                                                                                       
                
print(f'#train instances: {len(training_features)}')
print('first 3 feature dicts:', training_features[:3])
print('first 3 labels:', training_gold_labels[:3])

/Users/karina/Documents/labs-and-assignments/.venv312/lib/python3.12/site-packages/nltk/data.py:388: RuntimeWarning: Security Warning [pathsec.open]: Path /Users/karina/Documents/labs-and-assignments/lab_sessions/lab4/CONLL2003/CONLL2003/train.txt allowed via CWD.
  stream = _secure_open(self._path, "rb")


#train instances: 203621
first 3 feature dicts: [{'words': 'EU', 'pos': 'NNP'}, {'words': 'rejects', 'pos': 'VBZ'}, {'words': 'German', 'pos': 'JJ'}]
first 3 labels: ['B-ORG', 'O', 'B-MISC']


In [13]:
### Adapt the path to point to the CONLL2003 folder on your local machine
test_reader = ConllCorpusReader('CONLL2003/CONLL2003', 'test.txt', ['words', 'pos', 'ignore', 'chunk'])
                                                                                                                                                                
test_features = []
test_gold_labels = []                                                                                                                                           
                
for token, pos, ne_label in test_reader.iob_words():                                                                                                            
    a_dict = {
        'words': token,                                                                                                                                         
        'pos': pos,
    }
    test_features.append(a_dict)
    test_gold_labels.append(ne_label)
                                                                                                                                                                
print(f'#test instances: {len(test_features)}')
print('first 3 feature dicts:', test_features[:3])                                                                                                              
print('first 3 labels      :', test_gold_labels[:3])


#test instances: 46435
first 3 feature dicts: [{'words': 'SOCCER', 'pos': 'NN'}, {'words': '-', 'pos': ':'}, {'words': 'JAPAN', 'pos': 'NNP'}]
first 3 labels      : ['O', 'O', 'B-LOC']


/Users/karina/Documents/labs-and-assignments/.venv312/lib/python3.12/site-packages/nltk/data.py:388: RuntimeWarning: Security Warning [pathsec.open]: Path /Users/karina/Documents/labs-and-assignments/lab_sessions/lab4/CONLL2003/CONLL2003/test.txt allowed via CWD.
  stream = _secure_open(self._path, "rb")


**[2 points] b) provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [8]:
from collections import Counter 

my_list=[1,2,1,3,2,5]
Counter(my_list)


Counter({1: 2, 2: 2, 3: 1, 5: 1})

In [18]:
from collections import Counter                                                                                                                                                                             
                                                                                                                                                                                                            
print("Number of training instances:", len(training_gold_labels))                                                                                                                                           
print("Number of test instances:", len(test_gold_labels))                                                                                                                                                   
                                                                                                                                                                                                            
train_counts = Counter(training_gold_labels)
test_counts = Counter(test_gold_labels)                                                                                                                                                                     
                
print("\nLabel frequencies in training data:")                                                                                                                                                              
for label, count in train_counts.most_common():
    print(label, ":", count)                                                                                                                                                                                     
                
print("\nLabel frequencies in test data:")
for label, count in test_counts.most_common():
    print(label, ":", count) 

Number of training instances: 203621
Number of test instances: 46435

Label frequencies in training data:
O : 169578
B-LOC : 7140
B-PER : 6600
B-ORG : 6321
I-PER : 4528
I-ORG : 3704
B-MISC : 3438
I-LOC : 1157
I-MISC : 1155

Label frequencies in test data:
O : 38323
B-LOC : 1668
B-ORG : 1661
B-PER : 1617
I-PER : 1156
I-ORG : 835
B-MISC : 702
I-LOC : 257
I-MISC : 216


As can be seen from the outouts of our code cells above, the training set has 203,621 token instances and the test set has 46,435, so the test set is about 4-5 times smaller than the training set. Also, the label distributions are very unbalanced. In both train and test the non-entity tag O dominates: 169,578 out of 203,621 in train (around 83%) and 38,323 out of 46,435 in test (around 82%). 

Among the actual entity tags, B-LOC, B-PER and B-ORG are the most common, while I-LOC and I-MISC are by far the rarest, together accounting for less than 1.5% of all tokens. Training and test data follow a similar overall pattern (O dominates, B-tags are more frequent than I-tags), but the per-label proportions are not identical. For example B-LOC and B-ORG are almost tied in the test set (1668 vs 1661) while in the training set B-LOC is clearly more frequent than B-ORG (7140 vs 6321). B-MISC is also slightly less common in test (1.5%) than in train (1.7%). Because the data is so unbalanced, a classifier can reach high overall accuracy just by predicting O most of the time, so we need to look at precision, recall and F1 per class rather than overall accuracy.

**[2 points] c) Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.


In [9]:
from sklearn.feature_extraction import DictVectorizer

In [20]:
vec = DictVectorizer()
                                                                                                                                                                                                            
all_features = training_features + test_features                                                                                                                                                            
the_array = vec.fit_transform(all_features)                                                                                                                                                                 
                                                                                                                                                                                                            
n_train = len(training_features)                                                                                                                                                                            
train_array = the_array[:n_train]
test_array = the_array[n_train:]                                                                                                                                                                            
                
print("Combined matrix shape:", the_array.shape)                                                                                                                                                            
print("Train matrix shape:", train_array.shape)
print("Test matrix shape:", test_array.shape)                                                                                                                                                                  

Combined matrix shape: (250056, 27361)
Train matrix shape: (203621, 27361)
Test matrix shape: (46435, 27361)


**[4 points] d) Train the SVM using the train features and labels and evaluate on the test data. Provide a classification report (sklearn.metrics.classification_report).**
The train (*lin_clf.fit*) might take a while. On my computer, it took 1min 53s, which is acceptable. Training models normally takes much longer. If it takes more than 5 minutes, you can use a subset for training. Describe the results:
* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?

In [21]:
from sklearn import svm

In [25]:
# lin_clf = svm.LinearSVC()
lin_clf = svm.LinearSVC(max_iter=5000)  

In [26]:
##### [ YOUR CODE SHOULD GO HERE ]
# lin_clf.fit( # your code here
from sklearn.metrics import classification_report                                                                                                                                                           
                
lin_clf.fit(train_array, training_gold_labels)                                                                                                                                                              
predictions = lin_clf.predict(test_array)
                                                                                                                                                                                                            
print(classification_report(test_gold_labels, predictions, digits=3, zero_division=0))  

              precision    recall  f1-score   support

       B-LOC      0.812     0.775     0.793      1668
      B-MISC      0.782     0.664     0.718       702
       B-ORG      0.793     0.520     0.628      1661
       B-PER      0.860     0.437     0.579      1617
       I-LOC      0.618     0.529     0.570       257
      I-MISC      0.570     0.588     0.579       216
       I-ORG      0.703     0.467     0.561       835
       I-PER      0.332     0.871     0.481      1156
           O      0.985     0.984     0.985     38323

    accuracy                          0.920     46435
   macro avg      0.717     0.648     0.655     46435
weighted avg      0.939     0.920     0.923     46435



The classifier performs very well on the O tag, with precision, recall and F1 all around 0.985. This makes sense because O is teh most frequent class (about 83% of all tokens) and most O tokens are common function words and lowercase words that are easy to recognise. 

Among the entity tags, the model does best on B-LOC (with F1=0.793), B-MISC (F1=0.718). These are the first tokens of named entities, and so they usually have clear surface cues like capitalisation or country names that the (word, pos) features can pick up on quite easily. 

The classifier performs significantly worse on B-PER (F1 0.579), B-ORG and most of the I- tags. For B-PER and B-ORG the recall is much lower than the precision (0.437 and 0.520), which means the model is conservative: when it predicts B-PER it is usually right, but it misses many actual persons and organisations. This is probably because many person and organisation names are rare words that did not appear in the training data, so the lookup-style (word, pos) feature has nothing to match. 

The I- tags (I-LOC, I-ORG, I-MISC, I-PER) are the hardest. I-PER stands out with very high recall (0.871) but very low precision (0.332), meaning the model over-predicts I-PER and tags many tokens as part of a person name when they are not. This is a typical problem of a token-level classifier: deciding whether a word is "inside" a named entity really depends on the previous token, but our SVM looks at every token on its own and has no context. 
Overall accuracy is 0.92, but the macro-averaged F1 is only 0.655, which shows that teh high accuracy mostly comes from the dominant O class. The class imbalance from part (b) is clearly visible in the results: rare classes have low recall, and any tag whose meaning depends on neighbouring tokens (the I- tags) is hard to predict.

**[6 points] e) Train a model that uses the embeddings of these words as inputs. Test again on the same data as in 2d. Generate a classification report and compare the results with the classifier you built in 2d.**

In [ ]:
# your code here
import numpy as np
import gensim.downloader as api
word_embedding_model = api.load('word2vec-google-news-300')

def features_to_embeddings(features):
    matrix = np.zeros((len(features), 300))
    for i, f in enumerate(features):
        word = f['words']
        if word in word_embedding_model:
            matrix[i] = word_embedding_model[word]
    return matrix

train_embeddings = features_to_embeddings(training_features)
test_embeddings = features_to_embeddings(test_features)

print("Train embedding matrix:", train_embeddings.shape)
print("Test embedding matrix:", test_embeddings.shape)

emb_clf = svm.LinearSVC(max_iter=5000)
emb_clf.fit(train_embeddings, training_gold_labels)
emb_predictions = emb_clf.predict(test_embeddings)

print(classification_report(test_gold_labels, emb_predictions, digits=3, zero_division=0))

Train embedding matrix: (203621, 300)
Test embedding matrix: (46435, 300)
              precision    recall  f1-score   support

       B-LOC      0.759     0.801     0.779      1668
      B-MISC      0.724     0.695     0.709       702
       B-ORG      0.690     0.638     0.663      1661
       B-PER      0.746     0.669     0.705      1617
       I-LOC      0.514     0.424     0.465       257
      I-MISC      0.604     0.537     0.569       216
       I-ORG      0.480     0.332     0.392       835
       I-PER      0.586     0.501     0.540      1156
           O      0.973     0.991     0.982     38323

    accuracy                          0.927     46435
   macro avg      0.675     0.621     0.645     46435
weighted avg      0.921     0.927     0.923     46435



For this part we replaced the (word, pos) feature dictionary with the 300-dimensional Google News word2vec vector for each token. 

Words that are not in the embedding vocabulary get a zero vector. Then the same LinearSVC was trained on the train embedding matrix (203621, 300) and evaluated on the test embedding matrix (46435, 300). 

Here we saw the overall accuracy wnet up slightly compared to part (d), from 0.920 to 0.927, but the macro-averaged F1 actually drops a little (from 0.655 to 0.645). The weighted F1 is basically unchanged (0.923 in both). So the average looks similar, but the per class behaviour shifts a lot. 

- B-PER improves the most. F1 jumps from 0.579 to 0.705 and recall rises sharply from 0.437 to 0.669, a relative improvement of about 50%. (This is a clear win for the embeddings, since unseen surnames in the test set land close to other person names in the embedding space, while in (d) they were just unknown columns in the DictVectorizer.)
- B-ORG also improves (F1 0.628 to 0.663), with recall rising from 0.520 to 0.638 for the same reason. B-LOC and B-MISC stay at roughly the same F1 (0.793 to 0.779 for B-LOC and 0.718 to 0.709 for B-MISC). 
- For B-LOC the recall actually went up (0.775 to 0.801), so the model finds more locations, but precision dropped a bit. 

The strange behaviour of I-PER in part (d) is mostly fixed: in (d) the model over-predicted I-PER (precision 0.332, recall 0.871), and with embeddings this becomes a much more balanced precision 0.586 and recall 0.501, with F1 rising from 0.481 to 0.540. However the other I- tags become worse. 
- I-ORG drops the most: F1 falls from 0.561 to 0.392, and recall collapses from 0.467 to 0.332. I-LOC also drops (F1 0.570 to 0.465). (This is because the embedding tells us what a word means but not where it appears in a sequence: deciding whether a word is "inside" a named entity really depends on the previous token, and the embedding alone cannot model that.)
-  The O class stays really good in both models (F1 0.985 in (d), 0.982 in (e)), just because it is so dominant. In short, the word embeddings help a lot with unseen entity names (especially B-PER and B-ORG) and remove the worst weakness of the dictionary model (the I-PER over-prediction), but they do not solve the main problem of NERC: it is a sequence tagging task, and a token-level classifier without context cannot really model the I- tags. 
To get a clear improvement on those classes we would need to add context features (previous and next word, previous tag, word shape) or move to a sequence model like a CRF or a BiLSTM.

## [Points: 10] Exercise 2 (NERC): feature inspection using the [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)
**[6 points] a. Perform the same steps as in the previous exercise. Make sure you end up for both the training part (*df_train*) and the test part (*df_test*) with:**
* the features representation using **DictVectorizer**
* the NERC labels in a list

Please note that this is the same setup as in the previous exercise:
* load both train and test using:
    * list of dictionaries for features
    * list of NERC labels
* combine train and test features in a list and represent them using one hot encoding
* train using the training features and NERC labels

In [ ]:
import pandas

In [ ]:
##### Adapt the path to point to your local copy of NERC_datasets
path = 'ner_dataset.csv'
kaggle_dataset = pandas.read_csv(path, on_bad_lines='warn')

In [ ]:
len(kaggle_dataset)

In [ ]:
df_train = kaggle_dataset[:100000]
df_test = kaggle_dataset[100000:120000]
print(len(df_train), len(df_test))

**[4 points] b. Train and evaluate the model and provide the classification report:**
* use the SVM to predict NERC labels on the test data
* evaluate the performance of the SVM on the test data

Analyze the performance per NERC label.

## End of this notebook